# How accurately can we estimate unmeasured genotypes?

We assess imputation by temporarily hiding measured genotypes before phasing, estimating them from the remaining markers and reference panel, and comparing those estimates with the original calls. The hidden measurements provide a known answer against which to evaluate genotype recovery.

## Measurements reserved for validation

We reserve 10% of compatible array markers within predefined regions on chromosomes 1, 3, 13, 20, and 36. The regions include the original feasibility region, regions containing historical replication targets, and an additional chromosome without a predefined target. Region selection is fixed before inspecting the new imputation results.

At each reserved marker, the calls are hidden for all dogs before phasing. The masked analysis starts from the remaining unphased genotypes; it does not reuse phase estimated with the hidden calls present. The complete analysis retains the measured calls.

## Agreement and uncertainty

We compare measured and estimated genotype calls, estimated allele counts, and performance across allele-frequency groups. We also examine agreement among genotypes carrying the alternate allele, because high overall agreement can be dominated by common homozygous genotypes.

The software's reported accuracy estimates complement these comparisons; they do not replace them. Validation at array markers assesses recovery of those markers and does not independently establish phase accuracy or performance at every rare, sequence-only variant.

## Independence from the reference panel

We compare study dogs with reference samples using measured markers spread across the chromosomes. Very high agreement can flag possible sample overlap for review. Different sample names do not prove that the dogs are different, and this similarity screen does not establish that the study and reference dogs are unrelated.

Reference membership and relatedness should be considered when interpreting validation, because overlap can make estimated accuracy appear stronger.

## Recovery of hidden measurements

| Breed | Chromosome / region | Hidden markers | Called measurements | Genotype agreement | Pooled dosage correlation squared |
| --- | --- | ---: | ---: | ---: | ---: |
| GSP | chr1 | 61 | 7076 | 98.73% | 0.979 |
| LR | chr1 | 64 | 3069 | 98.92% | 0.985 |
| GSP | chr3 | 75 | 8695 | 99.28% | 0.991 |
| LR | chr3 | 74 | 3543 | 99.18% | 0.991 |
| GSP | chr13 | 70 | 8119 | 97.19% | 0.956 |
| LR | chr13 | 71 | 3404 | 95.42% | 0.933 |
| GSP | chr20 | 67 | 7767 | 97.32% | 0.960 |
| LR | chr20 | 68 | 3258 | 96.93% | 0.950 |
| GSP | chr36 | 61 | 7075 | 95.24% | 0.921 |
| LR | chr36 | 65 | 3117 | 95.06% | 0.927 |

These comparisons concern the predefined array-marker validation regions. They do not independently establish phase accuracy or reliability at every imputed sequence variant.

Accuracy varied across the predefined regions. Chromosomes 13 and 36 warrant closer review before variant selection. No study dogs crossed the predefined reference-similarity flag threshold; this does not establish absence of close relatives or independent reference membership.


## How much confidence should we place in an imputed marker?

We compared Beagle’s estimated accuracy with recovery of 676 array markers hidden before phasing. All measurable dosage correlations were positive, so this check found no reversed relationship between predicted and measured allele counts. The check covers these markers and regions; it cannot establish accuracy at every imputed variant.

| Breed | Minimum reported accuracy | Markers retained | Median measured accuracy | Retained markers below 0.80 measured accuracy |
| --- | ---: | ---: | ---: | ---: |
| GSP | 0.00 | 334 | 0.987 | 26 |
| GSP | 0.80 | 325 | 0.987 | 22 |
| GSP | 0.90 | 300 | 0.991 | 15 |
| GSP | 0.95 | 251 | 0.997 | 10 |
| GSP | 0.99 | 164 | 1.000 | 4 |
| LR | 0.00 | 342 | 0.988 | 36 |
| LR | 0.80 | 332 | 0.990 | 32 |
| LR | 0.90 | 302 | 0.992 | 20 |
| LR | 0.95 | 242 | 0.998 | 6 |
| LR | 0.99 | 139 | 1.000 | 3 |

Measured accuracy is the squared correlation between predicted allele dosage and the hidden genotype. The 0.80 comparison is a descriptive benchmark, not an adopted filter. Raising the reported-confidence threshold improves typical accuracy but leaves some poorly recovered markers. Even a threshold of 0.99 leaves four such GSP markers and three LR markers. These results do not support treating a confidence threshold alone as proof that a variant is reliable.

## Which historical findings can be tested directly?

Each breed retains 12 of the 18 predefined historical targets after array QC and reference alignment, and all 12 are present in its completed imputation output. The retained sets differ: the chromosome 6 target is retained in GSP, whereas the first chromosome 13 target is retained in LR. Six targets per breed were excluded during QC. Their exclusion is retained; imputation is not assumed to repair the reason for exclusion.

For primary replication, we recommend using the original QC-passed measured genotypes at the retained targets. Preserve any originally missing measurements as missing for this baseline. Imputed-only variants remain exploratory until further validation supports their use. Historical effect alleles and effect directions still need confirmation before directional replication can be assessed. Relatedness and the small number of behavioral cases must also be addressed in the association model.


## Reproduce the validation review

The inference procedure in Notebook 04 already creates the hidden-genotype outputs. This chapter reads those measurements, repeats the marker-level comparison when needed, and redraws the two validation figures. Its figure-only script calls write into a separate Version 2 result directory and do not rewrite either edition's notebooks. To review a separately reproduced inference run, set `imputation_folder` to the same directory selected in Notebook 04.

In [ ]:
from pathlib import Path
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, study_path, verify_datasets, run_script

In [ ]:
imputation_folder = study_root / "data/imputation/autosomal-20261003"
imputation = read_json(imputation_folder / "manifest.private.json")
assert imputation["status"] == "complete"
review_file = imputation_folder / "review/review.private.json"
if not review_file.exists():
    run_script(study_root, "review_imputation.py", "--run-dir", imputation_folder, linux=True)
review = read_json(review_file)
print("Hidden markers reviewed:", review["heldout_marker_count"])
print("Negative dosage correlations:", review["negative_correlations"])

### Draw the validation figures

The first plot compares measured allele frequency with recovery accuracy. The second compares Beagle's reported confidence with measured recovery. The plotting code is in [summarize_imputation.py](../scripts/summarize_imputation.py) and [plot_imputation_confidence.py](../scripts/plot_imputation_confidence.py).

In [ ]:
results_folder = study_root / "results/private/version2/05_imputation_validation"
run_script(study_root, "summarize_imputation.py", "--run-dir", imputation_folder,
           "--results-dir", results_folder, "--figures-only")
run_script(study_root, "plot_imputation_confidence.py", "--run-dir", imputation_folder, "--results-dir", results_folder,
           "--figures-only")

In [ ]:
from IPython.display import Image, display
for name in ["imputation_validation", "imputation_confidence"]:
    display(Image(filename=str(results_folder / "figures" / f"{name}.png")))

### Read the marker-level comparisons

This short function shows how the review counts markers below a measured-accuracy benchmark among those passing a reported-confidence cutoff. The benchmark describes these hidden array markers; it is not an adopted universal filter.

In [ ]:
markers = read_json(imputation_folder / "review/heldout_markers.private.json")

def compare_confidence(markers, breed, cutoff=0.95):
    kept = [m for m in markers if m["breed"] == breed
            and m["reported_DR2"] is not None and m["reported_DR2"] >= cutoff]
    measured = [m for m in kept if m["dosage_r2"] is not None]
    return {"retained": len(kept), "measurable": len(measured),
            "below_0.80": sum(m["dosage_r2"] < 0.80 for m in measured)}

for breed in ["GSP", "LR"]:
    print(breed, compare_confidence(markers, breed))